# Installing Dependencies


In [1]:
%pip install lightgbm xgboost scikit-learn numpy pandas


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

TRAIN_LABEL = train_labels
TEST_DATA   = test_labels

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)

Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)


## 1. Session Baselines + Dead EDA Detection

In [3]:
def compute_subject_baseline(sensor_df, val_col):
    out = {}
    for pid, grp in sensor_df.groupby('pid'):
        v = grp[val_col].dropna()
        out[pid] = (v.mean(), v.std() + 1e-8)
    return out

def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
            print(f'  {pid}: EDA zero ratio={zr:.1%} → DEAD')
    return dead

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

print('Train EDA dead sensors:')
TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
print('Test EDA dead sensors:')
TEST_EDA_DEAD  = eda_dead_subjects(testeda)
print(f'Train dead: {TRAIN_EDA_DEAD} | Test dead: {TEST_EDA_DEAD}')

Train EDA dead sensors:
  70N8: EDA zero ratio=99.6% → DEAD
  Y21H: EDA zero ratio=99.4% → DEAD
Test EDA dead sensors:
Train dead: {'Y21H', '70N8'} | Test dead: set()


## 2. Feature Extraction (+ HR ±15s, IBI ±15s windows)

In [4]:
WINDOWS_MS    = [2500, 5000, 10000]  # standard windows for all sensors
HR_EXTRA_MS   = 15000                # extra window for HR
TEMP_EXTRA_MS = 20000                # extra window for TEMP (worked in v19)
IBI_EXTRA_MS  = 15000                # extra window for IBI — reduces NaN rate
ROLL_WIN_MS   = 30000


def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
    else:
        for s in ['mean','std','range','slope','p25','p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def nan_eda_features(feat, wl):
    for key in list(feat.keys()):
        if f'eda_{wl}' in key and key not in [f'eda_{wl}_valid', f'eda_{wl}_zero_ratio']:
            feat[key] = np.nan
    return feat


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp,
                         sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid','timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        feat['bl_hr']   = bl_hr.get(pid,   (np.nan,1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan,1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan,1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan,1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan,1))[0]

        def get_win(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            # HR
            hr_v = get_win(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan

            # EDA with NaN masking
            eda_v = get_win(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
                feat[f'eda_{wl}_nz_frac'] = len(nz) / len(eda_v)
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, wl)
            else:
                for k in ['zero_ratio','valid','dev','nz_mean','nz_frac']:
                    feat[f'eda_{wl}_{k}'] = np.nan
                feat = nan_eda_features(feat, wl)

            # TEMP
            temp_v = get_win(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev'] = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan

            # IBI
            ibi_v = get_win(ibi_df, 'value', hw)
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            if len(ibi_v) >= 2:
                feat[f'ibi_{wl}_mean']  = np.mean(ibi_v)
                feat[f'ibi_{wl}_std']   = np.std(ibi_v)
                feat[f'ibi_{wl}_rmssd'] = np.sqrt(np.mean(np.diff(ibi_v)**2))
                feat[f'ibi_{wl}_dev']   = (np.mean(ibi_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','rmssd','dev']:
                    feat[f'ibi_{wl}_{s}'] = np.nan

            # ACC
            acc_v = get_win(acc_df, 'magnitude', hw)
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            if len(acc_v) >= 5:
                feat[f'acc_{wl}_mean']   = np.mean(acc_v)
                feat[f'acc_{wl}_std']    = np.std(acc_v)
                feat[f'acc_{wl}_energy'] = np.mean(acc_v**2)
                feat[f'acc_{wl}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','energy','dev']:
                    feat[f'acc_{wl}_{s}'] = np.nan

            # BVP
            bvp_v = get_win(bvp_df, 'value', hw)
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v,75) - np.percentile(bvp_v,25)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
            else:
                for s in ['std','range','iqr','dev']:
                    feat[f'bvp_{wl}_{s}'] = np.nan

            # EEG
            s_eeg = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts-hw) & (s_eeg.timestamp < ts+hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = win_eeg[col].mean()
                th = feat[f'eeg_theta_{wl}']
                la = feat[f'eeg_lowAlpha_{wl}']
                ha = feat[f'eeg_highAlpha_{wl}']
                lb = feat[f'eeg_lowBeta_{wl}']
                hb = feat[f'eeg_highBeta_{wl}']
                lg = feat[f'eeg_lowGamma_{wl}']
                de = feat[f'eeg_delta_{wl}']
                feat[f'eeg_theta_alpha_{wl}']  = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wl}']   = (lb + hb) / (la + ha + eps)
                feat[f'eeg_hbeta_lgamma_{wl}'] = hb / (lg + eps)
                feat[f'eeg_engage_{wl}']        = hb / (de + th + eps)
            else:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = np.nan
                for r in ['theta_alpha','beta_alpha','hbeta_lgamma','engage']:
                    feat[f'eeg_{r}_{wl}'] = np.nan

        # Extra HR window ±15s
        hr_v15 = get_win(hr_df, 'value', HR_EXTRA_MS)
        feat.update(win_stats(hr_v15, 'hr_w15s'))
        bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
        feat['hr_w15s_dev'] = (np.mean(hr_v15) - bl_m) / bl_s if len(hr_v15) >= 1 else np.nan

        # Extra TEMP window ±20s (carried from v19)
        temp_v20 = get_win(temp_df, 'value', TEMP_EXTRA_MS)
        feat.update(win_stats(temp_v20, 'temp_w20s'))
        bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
        feat['temp_w20s_dev'] = (np.mean(temp_v20) - bl_m) / bl_s if len(temp_v20) >= 1 else np.nan

        # Extra IBI window ±15s — reduces 55% NaN rate from narrow windows
        ibi_v15 = get_win(ibi_df, 'value', IBI_EXTRA_MS)
        bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
        if len(ibi_v15) >= 2:
            feat['ibi_w15s_mean']  = np.mean(ibi_v15)
            feat['ibi_w15s_std']   = np.std(ibi_v15)
            feat['ibi_w15s_rmssd'] = np.sqrt(np.mean(np.diff(ibi_v15)**2))
            feat['ibi_w15s_dev']   = (np.mean(ibi_v15) - bl_m) / bl_s
        else:
            for s in ['mean','std','rmssd','dev']:
                feat[f'ibi_w15s_{s}'] = np.nan

        # Rolling deviation (30s lookback)
        for sensor, df_, col in [('hr', hr_df, 'value'), ('temp', temp_df, 'value')]:
            s = df_[df_.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)][col].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)][col].values
            if len(past) >= 2 and len(cur) >= 1:
                feat[f'{sensor}_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat[f'{sensor}_roll_dev'] = np.nan

        if pid not in eda_dead_set:
            s = eda_df[eda_df.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)]['value'].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)]['value'].values
            if len(past) >= 2 and len(cur) >= 1:
                feat['eda_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat['eda_roll_dev'] = np.nan
        else:
            feat['eda_roll_dev'] = np.nan

        # Interaction features
        feat['hr_temp_product']    = feat.get('hr_w5s_mean', np.nan) * feat.get('temp_w5s_mean', np.nan)
        feat['dev_hr_eda_product'] = feat.get('hr_w5s_dev',  np.nan) * feat.get('eda_w5s_dev',  np.nan)

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction ready.')

Feature extraction ready.


In [5]:
print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)

Extracting TRAIN features...
Train features: (1456, 181)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)

Extracting TEST features...
Test features: (1496, 180)


## 3. Lag Features

In [7]:
LAG_BASE = ['hr_w5s_mean','hr_w5s_dev','hr_w15s_mean','hr_w15s_dev','hr_roll_dev',
            'eda_w5s_mean','eda_w5s_dev','eda_roll_dev',
            'temp_w5s_mean','temp_w5s_dev','temp_w20s_mean','temp_w20s_dev','temp_roll_dev',
            'ibi_w5s_mean','ibi_w15s_mean','ibi_w15s_rmssd',
            'bvp_w5s_std']
LAG_COLS = [c for c in LAG_BASE if c in train_feats.columns]


def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid','timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    for col in cols:
        df[f'{col}_roll3'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    return df


train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id','pid','timestamp','arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]
print(f'Total features: {len(FEAT_COLS)}')

Total features: 228


## 4. Training Setup

In [8]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, classification_report

train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_all)
TRAIN_PRIOR = np.bincount(y_all, minlength=5) / len(y_all)
print('Class weights:', {f'A{i+1}': round(w,2) for i,w in enumerate(cw)})
print('Training prior:', {f'A{i+1}': round(p,3) for i,p in enumerate(TRAIN_PRIOR)})

Class weights: {'A1': np.float64(5.29), 'A2': np.float64(0.68), 'A3': np.float64(0.53), 'A4': np.float64(0.84), 'A5': np.float64(4.04)}
Training prior: {'A1': np.float64(0.038), 'A2': np.float64(0.295), 'A3': np.float64(0.38), 'A4': np.float64(0.237), 'A5': np.float64(0.049)}


## 5. LOSO CV — LGB (10 seeds) + XGB (5 seeds)

In [9]:
import lightgbm as lgb
import xgboost as xgb

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS_LGB  = [42, 7, 123, 13, 99, 2024, 17, 55, 101, 256]  # 10 seeds
SEEDS_XGB  = [42, 7, 123, 13, 99]                           # 5 seeds

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_xgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float64)
loso_lgb, loso_xgb = [], []


def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=63, learning_rate=0.03,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=15, lambda_l1=0.3, lambda_l2=0.3,
        max_depth=7, verbose=-1, seed=seed, n_jobs=-1
    )


def get_xgb_params(seed):
    return dict(
        objective='multi:softprob', num_class=5, eval_metric='mlogloss',
        max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7,
        min_child_weight=10, reg_alpha=0.3, reg_lambda=0.3,
        seed=seed, verbosity=0, nthread=-1
    )


for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    sw_tr = cw[y_tr]

    f_lgb = np.zeros((va_mask.sum(), 5))
    f_xgb = np.zeros((va_mask.sum(), 5))
    t_lgb = np.zeros((len(test_feats), 5))
    t_xgb = np.zeros((len(test_feats), 5))

    for seed in SEEDS_LGB:
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr, num_boost_round=1500,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        f_lgb += m.predict(X_va)   / len(SEEDS_LGB)
        t_lgb += m.predict(X_test) / len(SEEDS_LGB)

    for seed in SEEDS_XGB:
        dtr_x = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
        dva_x = xgb.DMatrix(X_va, label=y_va)
        m_x = xgb.train(
            get_xgb_params(seed), dtr_x, num_boost_round=1500,
            evals=[(dva_x, 'val')],
            early_stopping_rounds=100, verbose_eval=False,
        )
        f_xgb += m_x.predict(dva_x).reshape(-1,5)               / len(SEEDS_XGB)
        t_xgb += m_x.predict(xgb.DMatrix(X_test)).reshape(-1,5) / len(SEEDS_XGB)

    oof_lgb[va_mask] = f_lgb
    oof_xgb[va_mask] = f_xgb
    test_lgb += t_lgb / len(TRAIN_PIDS)
    test_xgb += t_xgb / len(TRAIN_PIDS)

    ba_l = balanced_accuracy_score(y_va, f_lgb.argmax(axis=1))
    ba_x = balanced_accuracy_score(y_va, f_xgb.argmax(axis=1))
    loso_lgb.append(ba_l)
    loso_xgb.append(ba_x)
    print(f'  {fold_pid} — LGB: {ba_l:.4f} | XGB: {ba_x:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_lgb):.4f} ± {np.std(loso_lgb):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb):.4f} ± {np.std(loso_xgb):.4f}')

  01Z2 — LGB: 0.3667 | XGB: 0.3500
  70N8 — LGB: 0.1776 | XGB: 0.2903
  7PF3 — LGB: 0.1926 | XGB: 0.2049
  CQ2G — LGB: 0.2688 | XGB: 0.2652
  D1XP — LGB: 0.2179 | XGB: 0.2740
  DT5C — LGB: 0.2822 | XGB: 0.2800
  F1ZM — LGB: 0.1723 | XGB: 0.1274
  LIUY — LGB: 0.4986 | XGB: 0.5080
  SE4Q — LGB: 0.5064 | XGB: 0.4633
  TPQI — LGB: 0.0469 | XGB: 0.0700
  Y21H — LGB: 0.0819 | XGB: 0.0893

LOSO LGB mean: 0.2556 ± 0.1438
LOSO XGB mean: 0.2657 ± 0.1342


## 6. Blend — Fine Grid Search

In [10]:
best_ba, best_w = 0.0, 0.5
for w in np.arange(0.0, 1.01, 0.01):   # finer than v18's 0.05 steps
    blend = w * oof_lgb + (1-w) * oof_xgb
    ba    = balanced_accuracy_score(y_all, blend.argmax(axis=1))
    if ba > best_ba:
        best_ba, best_w = ba, w

print(f'Best blend LGB weight: {best_w:.2f} → OOF BA: {best_ba:.4f}')

oof_blend  = best_w * oof_lgb  + (1-best_w) * oof_xgb
test_blend = best_w * test_lgb + (1-best_w) * test_xgb
test_pred  = test_blend.argmax(axis=1) + 1

print('\nTest distribution:')
print(pd.Series(test_pred).value_counts().sort_index())
print('Expected from prior:', {f'A{i+1}': int(p*len(test_pred)) for i,p in enumerate(TRAIN_PRIOR)})

Best blend LGB weight: 0.20 → OOF BA: 0.2094

Test distribution:
1    110
2    313
3    706
4    286
5     81
Name: count, dtype: int64
Expected from prior: {'A1': 56, 'A2': 441, 'A3': 569, 'A4': 354, 'A5': 73}


## 7. OOF Diagnosis

In [11]:
oof_pred = oof_blend.argmax(axis=1)
print('=== OOF Classification Report ===')
print(classification_report(y_all, oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print('Per-subject LOSO BA:')
for pid, ba_l, ba_x in zip(TRAIN_PIDS, loso_lgb, loso_xgb):
    flag = ' ← LOW' if max(ba_l,ba_x) < 0.25 else ''
    print(f'  {pid}: LGB={ba_l:.4f} XGB={ba_x:.4f}{flag}')

print(f'\nFinal OOF BA : {balanced_accuracy_score(y_all, oof_pred):.4f}')
print(f'LOSO LGB mean: {np.mean(loso_lgb):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb):.4f}')

=== OOF Classification Report ===
              precision    recall  f1-score   support

   Arousal 1       0.06      0.07      0.07        55
   Arousal 2       0.28      0.36      0.31       430
   Arousal 3       0.32      0.29      0.31       554
   Arousal 4       0.43      0.32      0.37       345
   Arousal 5       0.00      0.00      0.00        72

    accuracy                           0.30      1456
   macro avg       0.22      0.21      0.21      1456
weighted avg       0.31      0.30      0.30      1456

Per-subject LOSO BA:
  01Z2: LGB=0.3667 XGB=0.3500
  70N8: LGB=0.1776 XGB=0.2903
  7PF3: LGB=0.1926 XGB=0.2049 ← LOW
  CQ2G: LGB=0.2688 XGB=0.2652
  D1XP: LGB=0.2179 XGB=0.2740
  DT5C: LGB=0.2822 XGB=0.2800
  F1ZM: LGB=0.1723 XGB=0.1274 ← LOW
  LIUY: LGB=0.4986 XGB=0.5080
  SE4Q: LGB=0.5064 XGB=0.4633
  TPQI: LGB=0.0469 XGB=0.0700 ← LOW
  Y21H: LGB=0.0819 XGB=0.0893 ← LOW

Final OOF BA : 0.2094
LOSO LGB mean: 0.2556
LOSO XGB mean: 0.2657


# Generating Final Submissions


In [12]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred,
})

submission.to_csv('submission-v20.csv', index=False)
print('submission-v20.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())

submission-v20.csv saved.
Shape: (1496, 2)
arousal
1    110
2    313
3    706
4    286
5     81
Name: count, dtype: int64
